# AI Video Engine — CPU Only

This notebook builds a real, playable short video entirely on CPU: script, neural voice-over, burned-in captions, procedural motion graphics, music, render, and quality gates.

**Run every cell in order, top to bottom.** Rerunning is safe: narration and scene renders are cached.

- Default output: 1280x720 @ 24 fps, H.264 + AAC, about 30 seconds.
- Set `AIVIDEO_SMOKE=1` before starting the kernel for a fast 640x360 test render.
- No API keys, GPU, or paid service required.

**Expected result:** `ai_video_output/cpu/final_video.mp4` plus `qa_report.json`. The QA status should be `REVIEW_REQUIRED` only because a human still has to review the creative.

## 1. Preflight
Detects OS, RAM, disk, FFmpeg, GPU, and Colab/Kaggle. Makes no changes.

In [ ]:
import os, sys, platform, shutil, subprocess, json, math, time, hashlib, io, re, random
import array, wave
from pathlib import Path

def _run(cmd):
    try:
        return subprocess.run(cmd, capture_output=True, text=True, timeout=25)
    except Exception as exc:
        return type("R", (), {"returncode": 1, "stdout": "", "stderr": str(exc)})()

def _first_line(text, fallback="unknown"):
    for line in (text or "").splitlines():
        if line.strip():
            return line.strip()
    return fallback

print("=" * 68)
print("AI VIDEO ENGINE — ENVIRONMENT PREFLIGHT")
print("=" * 68)
print("Python      :", sys.version.split()[0])
print("Platform    :", platform.platform())
print("Working dir :", Path.cwd())
print("CPU threads :", os.cpu_count())

ram_gb = None
try:
    if os.name == "nt":
        import ctypes
        class MEMORYSTATUSEX(ctypes.Structure):
            _fields_ = [("dwLength", ctypes.c_ulong), ("dwMemoryLoad", ctypes.c_ulong),
                        ("ullTotalPhys", ctypes.c_ulonglong), ("ullAvailPhys", ctypes.c_ulonglong),
                        ("ullTotalPageFile", ctypes.c_ulonglong), ("ullAvailPageFile", ctypes.c_ulonglong),
                        ("ullTotalVirtual", ctypes.c_ulonglong), ("ullAvailVirtual", ctypes.c_ulonglong),
                        ("ullAvailExtendedVirtual", ctypes.c_ulonglong)]
        m = MEMORYSTATUSEX(); m.dwLength = ctypes.sizeof(MEMORYSTATUSEX)
        ctypes.windll.kernel32.GlobalMemoryStatusEx(ctypes.byref(m))
        ram_gb = m.ullTotalPhys / 1e9
    else:
        ram_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9
except Exception:
    pass
print("RAM         :", f"{ram_gb:.1f} GB" if ram_gb else "unknown")
try:
    du = shutil.disk_usage(Path.cwd())
    print("Free disk   :", f"{du.free / 1e9:.1f} GB")
except Exception:
    print("Free disk   : unknown")

ffmpeg = shutil.which("ffmpeg")
ffprobe = shutil.which("ffprobe")
if ffmpeg:
    v = _first_line(_run([ffmpeg, "-version"]).stdout)
    print("FFmpeg      :", v)
else:
    print("FFmpeg      : NOT on PATH (imageio-ffmpeg fallback will be used)")
print("ffprobe     :", ffprobe or "NOT on PATH (basic fallback will be used)")

gpu_name, gpu_mem = None, None
if shutil.which("nvidia-smi"):
    r = _run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"])
    if r.returncode == 0 and r.stdout.strip():
        gpu_name, gpu_mem = r.stdout.strip().splitlines()[0].split(",", 1)
print("GPU         :", f"{gpu_name} ({gpu_mem.strip()})" if gpu_name else "none detected")

torch_cuda = False
try:
    import torch  # type: ignore
    torch_cuda = bool(torch.cuda.is_available())
except Exception:
    pass
print("CUDA/torch  :", "available" if torch_cuda else "not available (CPU path is fine)")

on_colab = bool(os.environ.get("COLAB_GPU") or Path("/content").exists() and os.environ.get("COLAB_RELEASE_TAG"))
on_kaggle = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or Path("/kaggle/working").exists())
print("Notebook env :", "Colab" if on_colab else ("Kaggle" if on_kaggle else "local / other"))
print("=" * 68)

## 2. Dependencies
Installs Pillow, edge-tts, and imageio-ffmpeg into the current kernel.

In [ ]:
import subprocess, sys, importlib.util

REQUIRED = {
    "PIL": "pillow>=10.0",
    "edge_tts": "edge-tts>=7.0",
    "imageio_ffmpeg": "imageio-ffmpeg>=0.5",
}
missing = [spec for mod, spec in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("Installing:", ", ".join(missing))
    r = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", *missing],
        capture_output=True, text=True,
    )
    if r.returncode != 0:
        print("WARNING: dependency install failed. Offline fallback may be limited.")
        print((r.stderr or "")[-500:])
else:
    print("Dependencies already present.")

for mod, spec in REQUIRED.items():
    ok = importlib.util.find_spec(mod) is not None
    print(f"  {mod:<18} {'OK' if ok else 'MISSING'}  ({spec})")
if importlib.util.find_spec("PIL") is None:
    raise RuntimeError("Pillow is required. Install it, then rerun this cell and the cells below.")

## 3. Configuration
`PROFILE` selects the hardware preset. `SCENES` is the creative manifest.

In [ ]:
import os, json
from pathlib import Path

PROFILE = "cpu"
SMOKE = os.environ.get("AIVIDEO_SMOKE", "0") == "1"

PRESETS = {
    "cpu":      dict(width=1280, height=720,  fps=24, crf=21, preset="veryfast", parallel=1),
    "cpu_gpu":  dict(width=1920, height=1080, fps=30, crf=19, preset="medium",   parallel=2),
    "tesla":    dict(width=1920, height=1080, fps=30, crf=18, preset="slow",     parallel=3),
}

CONFIG = {
    "profile": PROFILE,
    "voice": "en-US-AndrewNeural",
    "rate": "+2%",
    "gap": 0.35,
    "min_scene": 1.8,
    "seed": 7,
    "reset_cache": False,
    "output_root": os.environ.get("AIVIDEO_OUTPUT_ROOT", ""),
    "smoke": SMOKE,
}

def _default_root():
    explicit = CONFIG["output_root"].strip()
    if explicit:
        return Path(explicit).expanduser().resolve()
    if PROFILE == "tesla":
        if Path("/content").exists():
            return Path("/content/ai_video_output")
        if Path("/kaggle/working").exists():
            return Path("/kaggle/working/ai_video_output")
    return Path.cwd() / "ai_video_output"

def apply_profile():
    CONFIG.update(PRESETS[PROFILE])
    if SMOKE:
        CONFIG.update(width=640, height=360, fps=12, min_scene=1.1, gap=0.18)
        CONFIG["preset"] = "veryfast"
        CONFIG["parallel"] = 1
    root = _default_root()
    CONFIG["outdir"] = str(root / (PROFILE + ("_smoke" if SMOKE else "")))
    return CONFIG

SCENES = [
    {
        "id": "s1",
        "kicker": "AI VIDEO ENGINE",
        "title": "ONE NOTEBOOK.\nREAL VIDEOS.",
        "subtitle": "Run the cells. Ship a finished film.",
        "narration": "Your videos do not need a studio. They need a reliable production pipeline.",
        "palette": ("#050b18", "#123a63"),
        "accent": (0, 229, 255),
        "motif": "rings",
    },
    {
        "id": "s2",
        "kicker": "AUTOMATED",
        "title": "SCRIPT. VOICE.\nCAPTIONS. MOTION.",
        "subtitle": "Every layer is generated and verified.",
        "narration": "Every scene is rendered, captioned, and voiced automatically, then checked before export.",
        "palette": ("#160a1e", "#5b1d63"),
        "accent": (255, 92, 200),
        "motif": "waves",
    },
    {
        "id": "s3",
        "kicker": "HARDWARE AWARE",
        "title": "CPU OR GPU.\nYOUR CHOICE.",
        "subtitle": "Three profiles. Identical output contract.",
        "narration": "Three profiles cover CPU only laptops, consumer graphics cards, and Tesla class cloud runtimes.",
        "palette": ("#04140f", "#0d4d3a"),
        "accent": (54, 255, 166),
        "motif": "grid",
    },
    {
        "id": "s4",
        "kicker": "QUALITY GATE",
        "title": "NO BLACK FRAMES.\nNO SILENT AUDIO.",
        "subtitle": "The export is tested before you publish.",
        "narration": "Quality gates reject black frames, silent narration, broken streams, and failed decodes.",
        "palette": ("#1a0b06", "#6b2410"),
        "accent": (255, 158, 66),
        "motif": "bars",
    },
    {
        "id": "s5",
        "kicker": "DELIVER",
        "title": "PRESS PLAY.\nTHEN PUBLISH.",
        "subtitle": "A playable, production-ready file.",
        "narration": "When every check passes, the notebook hands you a playable file that is ready to publish.",
        "palette": ("#06081c", "#2b2f7a"),
        "accent": (140, 160, 255),
        "motif": "orbit",
    },
]

apply_profile()
print("Profile :", PROFILE, "(SMOKE test)" if SMOKE else "(full quality)")
print("Canvas  :", CONFIG["width"], "x", CONFIG["height"], "@", CONFIG["fps"], "fps")
print("Encoder :", "GPU if verified, else libx264")
print("Scenes  :", len(SCENES))
print("Output  :", CONFIG["outdir"])
print("Voice   :", CONFIG["voice"])
print("\nEdit SCENES to change the creative. Keep unique ids and non-empty narration.")

## 4. Optional providers
Disabled by default. The notebook never needs an API key to render.

In [ ]:
import os, json, urllib.request

LLM_BASE_URL = os.environ.get("LLM_BASE_URL", "").rstrip("/")
LLM_API_KEY = os.environ.get("LLM_API_KEY", "")
LLM_MODEL = os.environ.get("LLM_MODEL", "")

PROVIDERS = {
    "llm": bool(LLM_BASE_URL and LLM_API_KEY and LLM_MODEL),
    "tts": "edge-tts (default)", 
    "image": "procedural motion graphics (default)",
}
print("Optional provider status:", json.dumps(PROVIDERS, indent=2))
if not PROVIDERS["llm"]:
    print("LLM rewrite disabled. Set LLM_BASE_URL, LLM_API_KEY, LLM_MODEL to enable an OpenAI-compatible endpoint.")
    print("The video still renders fully offline after dependencies are installed.")

def _llm_json(prompt: str, timeout: int = 45) -> dict:
    if not PROVIDERS["llm"]:
        raise RuntimeError("LLM adapter is disabled")
    body = json.dumps({
        "model": LLM_MODEL,
        "temperature": 0.7,
        "response_format": {"type": "json_object"},
        "messages": [
            {"role": "system", "content": "You are a short-form video writer. Return strict JSON only."},
            {"role": "user", "content": prompt},
        ],
    }).encode("utf-8")
    req = urllib.request.Request(
        LLM_BASE_URL + "/chat/completions", data=body,
        headers={"Content-Type": "application/json", "Authorization": "Bearer " + LLM_API_KEY},
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        payload = json.loads(resp.read().decode("utf-8"))
    text = payload["choices"][0]["message"]["content"]
    return json.loads(text)

def rewrite_scenes_with_llm(scenes, brief="a 30 second promo for a single-notebook AI video pipeline"):
    """Optional: rewrite title/subtitle/narration. Structure, ids, colors, and motifs stay fixed."""
    if not PROVIDERS["llm"]:
        print("Skipped: LLM adapter disabled.")
        return scenes
    prompt = (
        "Rewrite these video scenes. Keep every id. Return JSON with key 'scenes', each item containing "
        "id, kicker, title (use \\n for a line break), subtitle, narration. Narration must be 8-18 spoken words. "
        f"Brief: {brief}. Input: {json.dumps([{k: s[k] for k in ('id','kicker','title','subtitle','narration')} for s in scenes])}"
    )
    try:
        data = _llm_json(prompt)
        by_id = {item["id"]: item for item in data.get("scenes", [])}
        out = []
        for s in scenes:
            item = by_id.get(s["id"], {})
            merged = dict(s)
            for key, limit in (("kicker", 28), ("subtitle", 90), ("narration", 220)):
                if isinstance(item.get(key), str) and item[key].strip():
                    merged[key] = item[key].strip()[:limit]
            if isinstance(item.get("title"), str) and item["title"].strip():
                merged["title"] = item["title"].strip()[:80]
            out.append(merged)
        print("LLM rewrite applied. Review the scenes cell output before rendering.")
        return out
    except Exception as exc:
        print(f"LLM rewrite failed ({exc.__class__.__name__}). Keeping the built-in script.")
        return scenes

# To enable: set env vars, then run:  SCENES = rewrite_scenes_with_llm(SCENES)

## 5. Utilities
FFmpeg discovery, probe fallback, fonts, text wrapping, caption chunking, logging.

In [ ]:
import hashlib, io, json, math, os, random, re, shutil, subprocess, time, wave
from pathlib import Path
from PIL import Image, ImageDraw, ImageFilter, ImageFont

OUT = Path(CONFIG["outdir"])
CACHE = OUT / "_cache"
LOGS = OUT / "logs"
for _d in (OUT, CACHE, LOGS):
    _d.mkdir(parents=True, exist_ok=True)
if CONFIG.get("reset_cache"):
    shutil.rmtree(CACHE, ignore_errors=True)
    CACHE.mkdir(parents=True, exist_ok=True)
    print("Cache reset.")

LOG_LINES = []

def log(msg):
    line = f"[{time.strftime('%H:%M:%S')}] {msg}"
    print(line)
    LOG_LINES.append(line)

def sha(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:16]

def _find_ffmpeg():
    ffmpeg = shutil.which("ffmpeg")
    ffprobe = shutil.which("ffprobe")
    if not ffmpeg:
        try:
            import imageio_ffmpeg
            ffmpeg = imageio_ffmpeg.get_ffmpeg_exe()
        except Exception:
            pass
    if not ffmpeg:
        raise RuntimeError("FFmpeg not found. Install FFmpeg or run: pip install imageio-ffmpeg")
    if not ffprobe or not Path(ffprobe).exists():
        candidate = Path(ffmpeg).with_name("ffprobe" + (".exe" if os.name == "nt" else ""))
        ffprobe = str(candidate) if candidate.exists() else (shutil.which("ffprobe") or "ffprobe")
    return ffmpeg, ffprobe

FFMPEG, FFPROBE = _find_ffmpeg()
log(f"FFmpeg: {FFMPEG}")

def run(cmd, capture=False, timeout=None):
    return subprocess.run(cmd, capture_output=capture, text=capture, timeout=timeout, check=False)

def probe(path):
    """ffprobe JSON, with a best-effort ffmpeg -i fallback if ffprobe is unavailable."""
    try:
        r = run([FFPROBE, "-v", "error", "-print_format", "json", "-show_format", "-show_streams", str(path)], capture=True)
        if r.returncode == 0 and (r.stdout or "").strip():
            return json.loads(r.stdout)
    except Exception:
        pass
    r = run([FFMPEG, "-hide_banner", "-i", str(path)], capture=True)
    text = (r.stderr or "") + "\n" + (r.stdout or "")
    info = {"format": {}, "streams": []}
    m = re.search(r"Duration:\s*(\d+):(\d+):(\d+(?:\.\d+)?)", text)
    if m:
        info["format"]["duration"] = str(int(m.group(1)) * 3600 + int(m.group(2)) * 60 + float(m.group(3)))
    for line in text.splitlines():
        if "Video:" in line and "Stream" in line:
            codec = re.search(r"Video:\s*([a-zA-Z0-9_]+)", line)
            dims = re.search(r"(\d{2,5})x(\d{2,5})", line)
            fps = re.search(r"([\d.]+)\s*fps", line)
            info["streams"].append({
                "codec_type": "video", "codec_name": codec.group(1) if codec else "unknown",
                "width": int(dims.group(1)) if dims else 0, "height": int(dims.group(2)) if dims else 0,
                "r_frame_rate": fps.group(1) if fps else "0/1",
            })
        elif "Audio:" in line and "Stream" in line:
            codec = re.search(r"Audio:\s*([a-zA-Z0-9_]+)", line)
            info["streams"].append({"codec_type": "audio", "codec_name": codec.group(1) if codec else "unknown"})
    return info

FONT_CANDIDATES = [
    "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
    "/usr/share/fonts/TTF/DejaVuSans-Bold.ttf",
    "/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf",
    "C:/Windows/Fonts/arialbd.ttf",
    "C:/Windows/Fonts/segoeuib.ttf",
    "/System/Library/Fonts/Supplemental/Arial Bold.ttf",
    "/usr/share/fonts/truetype/noto/NotoSans-Bold.ttf",
]
FONT_REG_CANDIDATES = [p.replace("-Bold", "") for p in FONT_CANDIDATES] + [
    "C:/Windows/Fonts/arial.ttf",
    "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
]
_FONTS = {}

def font(size, bold=True):
    key = ("b" if bold else "r", max(8, int(size)))
    if key in _FONTS:
        return _FONTS[key]
    chosen = None
    for p in (FONT_CANDIDATES if bold else FONT_REG_CANDIDATES):
        if Path(p).exists():
            chosen = p
            break
    if chosen:
        f = ImageFont.truetype(chosen, key[1])
    else:
        try:
            f = ImageFont.load_default(size=key[1])
        except TypeError:
            f = ImageFont.load_default()
    _FONTS[key] = f
    return f

def hex_rgb(value):
    value = value.lstrip("#")
    return tuple(int(value[i:i + 2], 16) for i in (0, 2, 4))

def mix(a, b, t):
    return tuple(int(a[i] * (1 - t) + b[i] * t) for i in range(3))

def wrap(draw, text, f, max_w):
    words, lines, cur = text.split(), [], ""
    for w in words:
        trial = f"{cur} {w}".strip()
        if draw.textlength(trial, font=f) <= max_w or not cur:
            cur = trial
        else:
            lines.append(cur)
            cur = w
    if cur:
        lines.append(cur)
    return lines

def caption_chunks(text, max_chars=82):
    words, chunks, cur = text.split(), [], ""
    for w in words:
        trial = f"{cur} {w}".strip()
        if len(trial) <= max_chars or not cur:
            cur = trial
        else:
            chunks.append(cur)
            cur = w
    if cur:
        chunks.append(cur)
    return chunks or [text]

def write_wav(path, pcm, channels=1, rate=44100):
    with wave.open(str(path), "wb") as wf:
        wf.setnchannels(channels)
        wf.setsampwidth(2)
        wf.setframerate(rate)
        wf.writeframes(pcm.tobytes())

## 6. Narration and timeline
Neural TTS per scene. Durations are measured, then frame-quantized so voice-over is never cut and audio/video cannot drift.

In [ ]:
RATE = 44100
TTS_STATUS = {"kind": "edge-tts", "fallback_scenes": [], "silent_scenes": []}

def pcm_rms(pcm):
    if not pcm:
        return 0.0
    step = max(1, len(pcm) // 200_000)
    total = 0
    count = 0
    for i in range(0, len(pcm), step):
        v = pcm[i]
        total += v * v
        count += 1
    return math.sqrt(total / max(1, count))

def tts_to_mp3(text, out, attempts=2):
    try:
        import asyncio
        import edge_tts
    except Exception as exc:
        log(f"edge-tts import failed: {exc}")
        return False
    async def go():
        communicate = edge_tts.Communicate(text, CONFIG["voice"], rate=CONFIG["rate"])
        await communicate.save(str(out))
    for attempt in range(1, attempts + 1):
        try:
            asyncio.run(go())
            if out.exists() and out.stat().st_size > 1200:
                return True
        except Exception as exc:
            log(f"TTS attempt {attempt} failed: {exc.__class__.__name__}: {exc}")
            time.sleep(1.0)
    return False

def fallback_voice(text, rate=RATE):
    """Deterministic synthetic narration bed. Used only when neural TTS is unavailable."""
    samples = array.array("h")
    words = text.replace(",", " ").replace(".", " ").split()
    for word in words:
        dur = 0.10 + min(0.16, 0.028 * len(word))
        f0 = 128 + (sum(map(ord, word)) % 36)
        for i in range(int(dur * rate)):
            t = i / rate
            env = min(1.0, t / 0.015) * min(1.0, (dur - t) / 0.03)
            v = 0.20 * math.sin(2 * math.pi * f0 * t) + 0.10 * math.sin(2 * math.pi * (f0 * 2.02) * t)
            samples.append(int(max(-1, min(1, v * env)) * 12000))
        samples.extend([0] * int(0.055 * rate))
    return samples

def decode_audio(path, rate=RATE):
    r = subprocess.run(
        [FFMPEG, "-v", "error", "-i", str(path), "-f", "s16le", "-ac", "1", "-ar", str(rate), "-"],
        capture_output=True, check=False,
    )
    if r.returncode != 0 or not r.stdout:
        raise RuntimeError(f"Could not decode narration {path}: {r.stderr[-300:]!r}")
    pcm = array.array("h")
    pcm.frombytes(r.stdout[: len(r.stdout) // 2 * 2])
    return pcm

log("Generating narration...")
scene_audio = {}
for scene in SCENES:
    key = sha(f"v1|{CONFIG['voice']}|{CONFIG['rate']}|{scene['narration']}")
    mp3 = CACHE / f"tts_{key}.mp3"
    if not mp3.exists() or mp3.stat().st_size < 1200:
        tts_to_mp3(scene["narration"], mp3)
    if mp3.exists() and mp3.stat().st_size >= 1200:
        try:
            pcm = decode_audio(mp3)
        except Exception as exc:
            log(f"  {scene['id']}: decode failed ({exc}); using fallback")
            pcm = fallback_voice(scene["narration"])
            TTS_STATUS["fallback_scenes"].append(scene["id"])
    else:
        pcm = fallback_voice(scene["narration"])
        TTS_STATUS["fallback_scenes"].append(scene["id"])
        mp3.unlink(missing_ok=True)
    rms = pcm_rms(pcm)
    if rms < 200:
        log(f"  {scene['id']}: silent TTS output (RMS {rms:.0f}); using fallback")
        pcm = fallback_voice(scene["narration"])
        TTS_STATUS["silent_scenes"].append(scene["id"])
        if scene["id"] not in TTS_STATUS["fallback_scenes"]:
            TTS_STATUS["fallback_scenes"].append(scene["id"])
        rms = pcm_rms(pcm)
    scene_audio[scene["id"]] = pcm
    log(f"  {scene['id']}: {len(pcm) / RATE:.2f}s narration, RMS {rms:.0f}")

FPS = int(CONFIG["fps"])
frames_for, scene_dur = {}, {}
for scene in SCENES:
    raw = len(scene_audio[scene["id"]]) / RATE
    seconds = max(CONFIG["min_scene"], raw + CONFIG["gap"])
    frames = int(math.ceil(seconds * FPS))
    frames_for[scene["id"]] = frames
    scene_dur[scene["id"]] = frames / FPS
TOTAL_FRAMES = sum(frames_for.values())
TOTAL_DUR = TOTAL_FRAMES / FPS
log(f"Timeline: {TOTAL_FRAMES} frames, {TOTAL_DUR:.2f}s at {FPS} fps")

voice = array.array("h")
for scene in SCENES:
    pcm = scene_audio[scene["id"]]
    target = int(round(scene_dur[scene["id"]] * RATE))
    voice.extend(pcm[:target])
    if len(pcm) < target:
        voice.extend([0] * (target - len(pcm)))
narration_wav = CACHE / "narration.wav"
write_wav(narration_wav, voice)
log(f"Narration track: {len(voice) / RATE:.2f}s -> {narration_wav}")
if TTS_STATUS["fallback_scenes"]:
    print("WARNING: synthetic fallback narration used for", TTS_STATUS["fallback_scenes"])

## 7. Visual system
Procedural backgrounds, kinetic typography, safe-area captions, and motion. No black frames are generated.

In [ ]:
W, H = int(CONFIG["width"]), int(CONFIG["height"])
SAFE_X = int(W * 0.06)
SAFE_TOP = int(H * 0.05)
CAPTION_W = int(W * 0.88)
CAPTION_Y = int(H * (0.74 if H > W else 0.755))
TITLE_SIZE = int(H * 0.082)
KICKER_SIZE = int(H * 0.026)
SUB_SIZE = int(H * 0.034)
CAP_SIZE = int(H * 0.032)
BRAND_SIZE = int(H * 0.023)

_title_font = font(TITLE_SIZE)
_kicker_font = font(KICKER_SIZE)
_sub_font = font(SUB_SIZE, bold=False)
_cap_font = font(CAP_SIZE)
_brand_font = font(BRAND_SIZE)
_measure = ImageDraw.Draw(Image.new("RGB", (10, 10)))

def background(scene, idx):
    c0, c1 = hex_rgb(scene["palette"][0]), hex_rgb(scene["palette"][1])
    sw, sh = max(8, W // 10), max(8, H // 10)
    small = Image.new("RGB", (sw, sh))
    px = small.load()
    for y in range(sh):
        for x in range(sw):
            t = (x / max(1, sw - 1)) * 0.55 + (y / max(1, sh - 1)) * 0.45
            px[x, y] = mix(c0, c1, t ** 1.25)
    base = small.resize((W, H), Image.Resampling.BILINEAR).convert("RGBA")

    rng = random.Random(CONFIG["seed"] * 101 + idx)
    for gi in range(3):
        g = Image.radial_gradient("L").resize((int(W * 0.8), int(H * 0.9)), Image.Resampling.BILINEAR)
        g = Image.eval(g, lambda v: max(0, 150 - v))
        color = scene["accent"] if gi == 0 else tuple(int(c * 0.55) for c in scene["accent"])
        glow = Image.new("RGBA", g.size, (*color, 0))
        glow.putalpha(g.point(lambda v: int(v * 0.55)))
        pos = (rng.randint(-int(W * 0.2), int(W * 0.6)), rng.randint(-int(H * 0.25), int(H * 0.55)))
        base.alpha_composite(glow, dest=(max(0, pos[0]), max(0, pos[1])))

    vig = Image.radial_gradient("L").resize((W, H), Image.Resampling.BILINEAR)
    vig = vig.point(lambda v: int(min(120, (v / 255) ** 2 * 165)))
    shade = Image.new("RGBA", (W, H), (0, 0, 0, 255))
    shade.putalpha(vig)
    base.alpha_composite(shade)
    return base

def draw_scene_frame(base, scene, idx, frame, nframes, total_index):
    img = base.copy()
    draw = ImageDraw.Draw(img, "RGBA")
    p = frame / max(1, nframes - 1)
    accent = scene["accent"]
    rng = random.Random(CONFIG["seed"] * 7919 + idx)

    sweep_w = int(W * 0.26)
    sweep_x = int(-sweep_w + p * (W + sweep_w * 2))
    band = Image.new("RGBA", (sweep_w, H), (*accent, 0))
    grad = Image.linear_gradient("L").resize((sweep_w, H), Image.Resampling.BILINEAR)
    band.putalpha(grad.point(lambda v: int(v * 0.18)))
    img.alpha_composite(band, dest=(sweep_x, 0))

    ov = Image.new("RGBA", (W, H), (0, 0, 0, 0))
    od = ImageDraw.Draw(ov)
    motif = scene["motif"]
    if motif == "rings":
        cx, cy = int(W * 0.78), int(H * 0.42)
        for r in range(6):
            rr = int((H * 0.10 + r * H * 0.045) * (1 + 0.05 * math.sin(p * math.tau + r)))
            od.ellipse((cx - rr, cy - rr, cx + rr, cy + rr), outline=(*accent, 70), width=max(2, H // 240))
    elif motif == "waves":
        for wv in range(3):
            pts = []
            for x in range(0, W + 12, 12):
                y = H * (0.38 + 0.07 * wv) + math.sin(x / W * math.tau * 1.5 + p * math.tau + wv) * H * 0.05
                pts.append((x, y))
            od.line(pts, fill=(*accent, 55), width=max(2, H // 300))
    elif motif == "grid":
        step = int(W * 0.055)
        off = int((p * step) % step)
        for x in range(-step + off, W + step, step):
            od.line([(x, 0), (x, H)], fill=(*accent, 26), width=1)
        for y in range(0, H, step):
            od.line([(0, y), (W, y)], fill=(*accent, 22), width=1)
    elif motif == "bars":
        for b in range(14):
            bh = int((0.10 + 0.34 * abs(math.sin(p * math.tau * 0.8 + b * 0.6))) * H)
            x = int(W * 0.62 + b * W * 0.026)
            od.rounded_rectangle((x, H - int(H * 0.30) - bh, x + int(W * 0.012), H - int(H * 0.30)), radius=4,
                                 fill=(*accent, 55))
    elif motif == "orbit":
        cx, cy = int(W * 0.78), int(H * 0.40)
        for o in range(3):
            rr = int(H * (0.12 + o * 0.05))
            od.ellipse((cx - rr, cy - rr * 0.55, cx + rr, cy + rr * 0.55), outline=(*accent, 60), width=2)
            ang = p * math.tau * (1 + o * 0.3) + o
            px, py = cx + math.cos(ang) * rr, cy + math.sin(ang) * rr * 0.55
            od.ellipse((px - 7, py - 7, px + 7, py + 7), fill=(*accent, 210))

    for k in range(16):
        px = int((rng.random() + p * (0.15 + 0.1 * rng.random())) % 1.0 * W)
        py = int((rng.random() - p * 0.08) % 1.0 * H)
        rad = 1 + int(2.5 * rng.random())
        od.ellipse((px - rad, py - rad, px + rad, py + rad), fill=(*accent, 60 + int(80 * rng.random())))
    img.alpha_composite(ov)

    if idx > 0 and frame < FPS // 3:
        q = frame / max(1, FPS // 3)
        wx = int(W * (1 - q))
        draw.rectangle((wx, 0, wx + int(W * 0.10), H), fill=(*accent, int(200 * (1 - q))))

    draw.text((SAFE_X, SAFE_TOP), "GENZO STUDIO", font=_brand_font, fill=(255, 255, 255, 205))
    draw.rectangle((SAFE_X, SAFE_TOP + int(H * 0.040), SAFE_X + int(W * 0.03), SAFE_TOP + int(H * 0.045)),
                   fill=(*accent, 230))
    label = f"{idx + 1:02d} / {len(SCENES):02d}"
    tw = draw.textlength(label, font=_brand_font)
    draw.text((W - SAFE_X - tw, SAFE_TOP), label, font=_brand_font, fill=(255, 255, 255, 160))

    float_y = int(math.sin(p * math.tau * 0.8) * H * 0.006)
    y = int(H * 0.245) + float_y
    draw.text((SAFE_X + 2, y + 2), scene["kicker"].upper(), font=_kicker_font, fill=(0, 0, 0, 160))
    draw.text((SAFE_X, y), scene["kicker"].upper(), font=_kicker_font, fill=(*accent, 255))

    for line in scene["title"].split("\n"):
        draw.text((SAFE_X + 3, y + int(H * 0.055) + 3), line, font=_title_font, fill=(0, 0, 0, 190))
        draw.text((SAFE_X, y + int(H * 0.055)), line, font=_title_font, fill=(255, 255, 255, 255))
        y += int(TITLE_SIZE * 1.08)

    sub_y = max(y + int(H * 0.018), int(H * 0.560))
    for line in wrap(draw, scene["subtitle"], _sub_font, int(W * 0.62)):
        draw.text((SAFE_X, sub_y), line, font=_sub_font, fill=(220, 230, 245, 215))
        sub_y += int(SUB_SIZE * 1.28)

    chunks = caption_chunks(scene["narration"])
    weights = [len(c) for c in chunks]
    total_w = sum(weights)
    acc = 0.0
    active = chunks[-1]
    for chunk, weight in zip(chunks, weights):
        acc += weight / total_w
        if p <= acc + 1e-9:
            active = chunk
            break
    lines = wrap(draw, active, _cap_font, CAPTION_W - int(W * 0.05))
    line_h = int(CAP_SIZE * 1.32)
    box_h = line_h * len(lines) + int(H * 0.035)
    bx, by = SAFE_X, CAPTION_Y
    draw.rounded_rectangle((bx, by, bx + CAPTION_W, by + box_h), radius=int(H * 0.018),
                           fill=(4, 9, 20, 205), outline=(*accent, 170), width=max(1, H // 480))
    draw.rectangle((bx + int(W * 0.012), by + int(H * 0.012), bx + int(W * 0.016), by + box_h - int(H * 0.012)),
                   fill=(*accent, 255))
    ty = by + int(H * 0.018)
    for line in lines:
        draw.text((bx + int(W * 0.028), ty), line, font=_cap_font, fill=(255, 255, 255, 245))
        ty += line_h

    gp = (total_index + frame) / max(1, TOTAL_FRAMES)
    draw.rectangle((0, H - 5, W, H), fill=(255, 255, 255, 28))
    draw.rectangle((0, H - 5, int(W * gp), H), fill=(*accent, 230))
    return img.convert("RGB")

print("Visual system ready:", W, "x", H)

## 8. Render scenes
Streams raw RGB frames into FFmpeg. Cached per scene, parallel when the profile allows.

In [ ]:
def pick_encoder(allow_gpu):
    if allow_gpu:
        enc = run([FFMPEG, "-hide_banner", "-encoders"], capture=True)
        text = (enc.stdout or "") + (enc.stderr or "")
        if "h264_nvenc" in text:
            test = CACHE / "nvenc_probe.mp4"
            r = run([FFMPEG, "-y", "-v", "error", "-f", "lavfi", "-i", "color=c=black:s=320x240:d=0.2",
                     "-c:v", "h264_nvenc", "-pix_fmt", "yuv420p", str(test)], capture=True)
            if r.returncode == 0 and test.exists() and test.stat().st_size > 1000:
                return "h264_nvenc", ["-rc", "vbr", "-cq", "23", "-preset", "p5"]
            log("NVENC probe failed; falling back to libx264")
    return "libx264", ["-crf", str(CONFIG["crf"]), "-preset", CONFIG["preset"]]

VENC, VENC_ARGS = pick_encoder(allow_gpu=(PROFILE != "cpu"))
log(f"Video encoder: {VENC}")

def render_scene(scene, idx):
    nframes = frames_for[scene["id"]]
    key = sha(json.dumps([scene, CONFIG["width"], CONFIG["height"], FPS, nframes, VENC, CONFIG["seed"]]))
    out = CACHE / f"scene_{idx:02d}_{key}.mp4"
    if out.exists() and out.stat().st_size > 10_000:
        log(f"  {scene['id']}: cached")
        return out
    base = background(scene, idx)
    tmp = out.with_suffix(".part.mp4")
    cmd = [FFMPEG, "-y", "-v", "error", "-f", "rawvideo", "-pix_fmt", "rgb24", "-s", f"{W}x{H}",
           "-r", str(FPS), "-i", "-", "-an", "-c:v", VENC, *VENC_ARGS, "-pix_fmt", "yuv420p",
           "-movflags", "+faststart", str(tmp)]
    proc = subprocess.Popen(cmd, stdin=subprocess.PIPE, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE)
    assert proc.stdin is not None
    t0 = time.time()
    try:
        for f in range(nframes):
            total_index = sum(frames_for[s["id"]] for s in SCENES[:idx])
            proc.stdin.write(draw_scene_frame(base, scene, idx, f, nframes, total_index).tobytes())
        proc.stdin.close()
        err = proc.stderr.read().decode("utf-8", "replace") if proc.stderr else ""
        rc = proc.wait()
    except BrokenPipeError:
        err = proc.stderr.read().decode("utf-8", "replace") if proc.stderr else ""
        rc = proc.wait()
    if rc != 0 or not tmp.exists() or tmp.stat().st_size < 10_000:
        raise RuntimeError(f"Scene render failed ({scene['id']}): {err[-600:]}")
    tmp.replace(out)
    log(f"  {scene['id']}: rendered {nframes} frames in {time.time() - t0:.1f}s")
    return out

log("Rendering scenes...")

def _safe_workers(requested):
    total_gb = 0.0
    try:
        if os.name != "nt":
            total_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9
        else:
            import ctypes
            class _MEM(ctypes.Structure):
                _fields_ = [("dwLength", ctypes.c_ulong), ("dwMemoryLoad", ctypes.c_ulong),
                            ("ullTotalPhys", ctypes.c_ulonglong), ("ullAvailPhys", ctypes.c_ulonglong),
                            ("ullTotalPageFile", ctypes.c_ulonglong), ("ullAvailPageFile", ctypes.c_ulonglong),
                            ("ullTotalVirtual", ctypes.c_ulonglong), ("ullAvailVirtual", ctypes.c_ulonglong),
                            ("ullAvailExtendedVirtual", ctypes.c_ulonglong)]
            m = _MEM(); m.dwLength = ctypes.sizeof(_MEM)
            ctypes.windll.kernel32.GlobalMemoryStatusEx(ctypes.byref(m))
            total_gb = m.ullTotalPhys / 1e9
    except Exception:
        return requested
    if total_gb and total_gb < 4:
        return 1
    if total_gb and total_gb < 8:
        return min(requested, 2)
    return requested

workers = _safe_workers(max(1, int(CONFIG["parallel"])))
if workers != int(CONFIG["parallel"]):
    log(f"Memory guard: capping parallel workers {CONFIG['parallel']} -> {workers}")
if workers > 1 and len(SCENES) > 1:
    from concurrent.futures import ThreadPoolExecutor
    log(f"Parallel scene rendering with {workers} workers")
    with ThreadPoolExecutor(max_workers=workers) as pool:
        scene_files = list(pool.map(lambda pair: render_scene(pair[1], pair[0]), enumerate(SCENES)))
else:
    scene_files = [render_scene(scene, i) for i, scene in enumerate(SCENES)]

concat_list = OUT / "concat.txt"
concat_list.write_text("".join(f"file '{p.as_posix()}'\n" for p in scene_files), encoding="utf-8")
video_only = OUT / "video_only.mp4"
r = run([FFMPEG, "-y", "-v", "error", "-f", "concat", "-safe", "0", "-i", str(concat_list),
         "-c", "copy", "-movflags", "+faststart", str(video_only)], capture=True)
if r.returncode != 0:
    raise RuntimeError(f"Concat failed: {r.stderr[-500:]}")
log("Video track assembled")

## 9. Audio mix and final mux
Deterministic music bed, voice-first mix, loudness normalization, H.264/AAC export.

In [ ]:
music_path = CACHE / "music.wav"
if not music_path.exists():
    total = int(TOTAL_DUR * RATE)
    chords = [(220.0, 261.63, 329.63), (174.61, 220.0, 261.63), (130.81, 196.0, 261.63), (196.0, 246.94, 293.66)]
    out = array.array("h")
    for n in range(total):
        t = n / RATE
        chord = chords[int(t / max(0.001, TOTAL_DUR / 4)) % 4]
        env = min(1.0, t / 1.2, max(0.0, (TOTAL_DUR - t) / 1.2))
        lfo = 0.75 + 0.25 * math.sin(2 * math.pi * 0.12 * t)
        val = 0.055 * env * lfo * sum(math.sin(2 * math.pi * f * t) for f in chord) / 3
        val += 0.015 * env * math.sin(2 * math.pi * 55 * t)
        out.append(int(max(-1, min(1, val)) * 32767))
    write_wav(music_path, out)
log("Music bed ready")

audio_m4a = OUT / "audio.m4a"
r = run([FFMPEG, "-y", "-v", "error", "-i", str(narration_wav), "-i", str(music_path),
         "-filter_complex",
         "[0:a]aresample=44100,volume=1.0[v];[1:a]aresample=44100,volume=0.9[m];"
         "[v][m]amix=inputs=2:duration=first:normalize=0,"
         "loudnorm=I=-16:TP=-1.5:LRA=11[a]",
         "-map", "[a]", "-c:a", "aac", "-b:a", "192k", "-ar", "44100", str(audio_m4a)], capture=True)
if r.returncode != 0:
    raise RuntimeError(f"Audio mix failed: {r.stderr[-500:]}")
log("Audio mixed")

final = OUT / "final_video.mp4"
r = run([FFMPEG, "-y", "-v", "error", "-i", str(video_only), "-i", str(audio_m4a),
         "-map", "0:v:0", "-map", "1:a:0", "-c:v", "copy", "-c:a", "aac", "-b:a", "192k",
         "-shortest", "-movflags", "+faststart", str(final)], capture=True)
if r.returncode != 0:
    raise RuntimeError(f"Mux failed: {r.stderr[-500:]}")
log(f"Final export: {final} ({final.stat().st_size / 1e6:.1f} MB)")
print("Run the next cell to execute the quality gates.")

## 10. Quality gates
Rejects black segments, silent narration, decode errors, and wrong metadata.

In [ ]:
def qa(path):
    report = {"file": str(path), "checks": [], "warnings": [], "status": "PASS", "measured": {}}
    info = probe(path)
    fmt = info.get("format", {})
    streams = info.get("streams", [])
    vs = next((s for s in streams if s.get("codec_type") == "video"), None)
    aus = next((s for s in streams if s.get("codec_type") == "audio"), None)
    dur = float(fmt.get("duration", 0) or 0)
    report["measured"].update({
        "duration": round(dur, 3), "size_bytes": path.stat().st_size,
        "video_codec": (vs or {}).get("codec_name"), "audio_codec": (aus or {}).get("codec_name"),
        "width": (vs or {}).get("width"), "height": (vs or {}).get("height"),
        "fps": (vs or {}).get("r_frame_rate"), "streams": len(streams),
    })

    def check(name, ok, detail, skipped=False):
        entry = {"name": name, "pass": bool(ok), "detail": detail}
        if skipped:
            entry = {"name": name, "pass": False, "skipped": True, "detail": detail}
            report["warnings"].append(f"Skipped: {name} ({detail})")
            return
        report["checks"].append(entry)
        if not ok:
            report["status"] = "FAIL"

    check("video stream exists", vs is not None, f"codec={(vs or {}).get('codec_name')}")
    check("audio stream exists", aus is not None, f"codec={(aus or {}).get('codec_name')}")
    check("duration matches timeline", abs(dur - TOTAL_DUR) < 0.35, f"{dur:.2f}s vs {TOTAL_DUR:.2f}s")
    check("resolution matches profile", bool(vs) and vs.get("width") == W and vs.get("height") == H,
          f"{vs.get('width')}x{vs.get('height')}" if vs else "n/a")
    min_bytes = 60_000 if SMOKE else 120_000
    check("file is non-trivial", path.stat().st_size > min_bytes, f"{path.stat().st_size} bytes")

    dec = run([FFMPEG, "-v", "error", "-i", str(path), "-f", "null", "-"], capture=True, timeout=600)
    check("full decode without errors", dec.returncode == 0 and not (dec.stderr or "").strip(),
          "clean" if not (dec.stderr or "").strip() else (dec.stderr or "")[-200:])

    bd = run([FFMPEG, "-hide_banner", "-i", str(path), "-vf", "blackdetect=d=0.2:pix_th=0.10",
              "-an", "-f", "null", "-"], capture=True, timeout=600)
    black = [ln for ln in (bd.stderr or "").splitlines() if "black_start" in ln]
    check("no black segment", not black, "none" if not black else black[0][:160])

    vd = run([FFMPEG, "-hide_banner", "-i", str(path), "-af", "volumedetect", "-f", "null", "-"],
             capture=True, timeout=600)
    vtxt = vd.stderr or ""
    maxv = meanv = None
    for ln in vtxt.splitlines():
        if "max_volume:" in ln:
            maxv = float(ln.split("max_volume:")[1].split("dB")[0])
        if "mean_volume:" in ln:
            meanv = float(ln.split("mean_volume:")[1].split("dB")[0])
    report["measured"]["max_volume_db"] = maxv
    report["measured"]["mean_volume_db"] = meanv
    check("audio is audible", maxv is not None and maxv > -30, f"max={maxv} dB mean={meanv} dB")

    sd = run([FFMPEG, "-hide_banner", "-i", str(path), "-af", "silencedetect=noise=-38dB:d=0.6",
              "-f", "null", "-"], capture=True, timeout=600)
    silences = [ln for ln in (sd.stderr or "").splitlines() if "silence_duration" in ln]
    silent_total = sum(float(ln.split("silence_duration:")[1].split()[0]) for ln in silences)
    report["measured"]["silent_seconds"] = round(silent_total, 2)
    check("final audio not mostly silent", silent_total < max(2.0, TOTAL_DUR * 0.45),
          f"{silent_total:.2f}s silence / {TOTAL_DUR:.2f}s")

    energy = {sid: round(pcm_rms(pcm), 1) for sid, pcm in scene_audio.items()}
    ndur = {sid: round(len(pcm) / RATE, 2) for sid, pcm in scene_audio.items()}
    report["measured"]["narration_rms"] = energy
    report["measured"]["narration_seconds"] = ndur
    check("every scene has voice audio",
          all(d >= 0.5 for d in ndur.values()) and all(v >= 200 for v in energy.values()),
          f"durations={ndur} rms={energy}")

    nv = run([FFMPEG, "-hide_banner", "-i", str(narration_wav), "-af", "volumedetect", "-f", "null", "-"],
             capture=True, timeout=300)
    nmax = None
    for ln in (nv.stderr or "").splitlines():
        if "max_volume:" in ln:
            nmax = float(ln.split("max_volume:")[1].split("dB")[0])
    report["measured"]["narration_max_volume_db"] = nmax
    check("narration track is audible", nmax is not None and nmax > -30, f"max={nmax} dB")

    bright = []
    for frac in (0.03, 0.25, 0.5, 0.75, 0.97):
        ts = max(0.0, min(TOTAL_DUR - 0.1, TOTAL_DUR * frac))
        r = subprocess.run([FFMPEG, "-v", "error", "-ss", f"{ts:.3f}", "-i", str(path), "-frames:v", "1",
                            "-f", "image2pipe", "-vcodec", "png", "-"], capture_output=True, check=False)
        if r.returncode == 0 and r.stdout:
            im = Image.open(io.BytesIO(r.stdout)).convert("L")
            bright.append(round(sum(im.resize((64, 36)).getdata()) / (64 * 36), 1))
    report["measured"]["sample_brightness"] = bright
    check("frames are not black (sampled)", bool(bright) and min(bright) > 12, f"luma samples={bright}")

    check("scene count and captions valid", len(SCENES) == len(scene_files) == 5 and
          all(caption_chunks(s["narration"]) for s in SCENES), f"{len(SCENES)} scenes")

    if TTS_STATUS["fallback_scenes"]:
        report["warnings"].append(f"Synthetic fallback narration used for {TTS_STATUS['fallback_scenes']}")
        if report["status"] == "PASS":
            report["status"] = "REVIEW_REQUIRED"
    if TTS_STATUS.get("silent_scenes"):
        report["warnings"].append(f"Silent TTS output detected for {TTS_STATUS['silent_scenes']}; fallback used")
        if report["status"] == "PASS":
            report["status"] = "REVIEW_REQUIRED"
    report["warnings"].append("Creative review (story, typography, pacing, rights) still requires a human.")
    if report["status"] == "PASS":
        report["status"] = "REVIEW_REQUIRED"
    return report

report = qa(final)
(OUT / "qa_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
print("\nQA RESULT:", report["status"])
for c in report["checks"]:
    print(f"  [{'PASS' if c['pass'] else 'FAIL'}] {c['name']}: {c['detail']}")
for w in report["warnings"]:
    print("  [WARN]", w)
print("\nMeasured:", json.dumps(report["measured"], indent=2))
if report["status"] == "FAIL":
    print("\nSTOP: fix the failed checks before publishing. The file is kept for inspection.")
else:
    print("\nTechnical gates passed. Run the playback cell, then do the creative review.")

## 11. Play it
Inline player plus a direct download link.

In [ ]:
from pathlib import Path
final_path = Path(final)
print("File :", final_path)
print("Size :", f"{final_path.stat().st_size / 1e6:.2f} MB")
print("Path :", final_path.resolve())
try:
    from IPython.display import Video, display, HTML
    if final_path.stat().st_size < 40_000_000:
        display(Video(str(final_path), embed=True, width=min(960, int(CONFIG["width"]))))
    else:
        display(HTML(f'<video controls src="{final_path.as_uri()}" style="max-width:960px"></video>'))
    display(HTML(f'<p><a href="{final_path.as_uri()}" download>Download {final_path.name}</a></p>'))
except Exception as exc:
    print("Inline player unavailable here:", exc)
    print("Open the file directly:", final_path.resolve())
try:
    if os.environ.get("AIVIDEO_AUTODOWNLOAD", "0") == "1":
        from google.colab import files
        files.download(str(final_path))
except Exception:
    pass

## 12. Production report
Paths, measurements, cache state, and the remaining human-review items.

In [ ]:
summary = {
    "profile": PROFILE,
    "smoke_test": SMOKE,
    "status": report["status"],
    "encoder": VENC,
    "tts": TTS_STATUS,
    "frames": TOTAL_FRAMES,
    "fps": FPS,
    "duration_seconds": round(TOTAL_DUR, 3),
    "resolution": f"{W}x{H}",
    "output": str(Path(final).resolve()),
    "qa_report": str((OUT / "qa_report.json").resolve()),
    "cache_dir": str(CACHE.resolve()),
}
(OUT / "job_state.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
(LOGS / "render.log").write_text("\n".join(LOG_LINES), encoding="utf-8")

print("=" * 68)
print("PRODUCTION REPORT")
print("=" * 68)
for k, v in summary.items():
    print(f"{k:>18}: {v}")
print("-" * 68)
print("Verified automatically:")
for c in report["checks"]:
    print(f"  [{'PASS' if c['pass'] else 'FAIL'}] {c['name']}")
print("Pending human review:")
for w in report["warnings"]:
    print("  -", w)
print("-" * 68)
print("Next: watch the video above. If it looks right, edit SCENES and rerun the")
print("narration cell onward. Cached assets are reused; set CONFIG['reset_cache']")
print("= True (or change the narration text) to force new renders.")
print("=" * 68)